> **Release dependency:** the original `clinical_review/` helper package was absent from the supplied source tree. This notebook is retained for documentation but cannot run until those original helpers are restored. See `docs/REPRODUCIBILITY.md`. No completed reviewer data are distributed.

# BR07S — Prepare Shaoxing images for one human reviewer

Run after Shaoxing BR04B generation and BR04C archival. **BR05A scoring is not a prerequisite.** Uses the existing Biowulf `clinical_review.workflow` and preserves its reference-before-report, blinding, coverage and adjudication gates. No model loading or new rendering. This does not complete the separate diagnosis-code mapping review.

Default is 10 ECGs × 2 models × 2 arms = 40 reports, calibration only. Change settings before preparation for a prespecified main review. Download only PUBLIC; keep PRIVATE away from the reviewer. Upload the accompanying `SHAOXING_HUMAN_REVIEW_INSTRUCTIONS.txt` beside this notebook.


In [ ]:
from pathlib import Path
import sys, json
from unittest.mock import patch
EXP = Path('/path/to/repository/experiment02')
CONFIG_PATH = Path('/path/to/repository/outputs/real_run_001/experiment02_v1/context_fit/v1/shaoxing/config.json')
if not (EXP / 'clinical_review/workflow.py').is_file():
    raise FileNotFoundError('Restore the existing clinical_review/workflow.py used by BR07–BR09 on Biowulf; do not replace it with another workflow.')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read, sha, immutable, lock
CFG = load_config(CONFIG_PATH)
assert CFG['cohort'] == 'shaoxing', 'Use the Shaoxing child config'
parent_setup(CFG)
RUN = Path(CFG['run_dir'])
from clinical_review.workflow import prepare, progress
from generation import folder_for
from extensions.query_image_archive import verified_provider, archive_folder
import queries
GUIDE = EXP / 'SHAOXING_HUMAN_REVIEW_INSTRUCTIONS.txt'
if not GUIDE.is_file(): raise FileNotFoundError(str(GUIDE))
print('Cohort:', CFG['cohort'], '| run:', RUN)


In [ ]:
REVIEW_VERSION = 'calibration_v1'  # Fresh version for a main review; never overwrite calibration.
REVIEW_DIR = RUN / 'clinical_review' / 'shaoxing' / REVIEW_VERSION
SELECTED_MODELS = ('llama_ecg', 'qwen38')
TOKEN_LIMITS = {'pulse': 384, 'llama_ecg': 384, 'llama_base': 1024, 'qwen38': 1024}
FOLDERS = {m: folder_for(CFG, m, False, TOKEN_LIMITS[m]) for m in SELECTED_MODELS}
ARMS = ('bidirectional_pool', 'bidirectional_graph')
# To review all methods, set ARMS before preparation to:
# ('sensor_only', 'forward_pool', 'bidirectional_pool', 'bidirectional_graph', 'bidirectional_shuffled_graph')
N_QUERIES = 10
STAGE = 'calibration'
REVIEWER_ID = 'ECG_reviewer_01'
REVIEWER_EXPERTISE = ''  # Required: specialty, years interpreting ECGs, actual study-development involvement.
SAMPLE_SIZE_JUSTIFICATION = 'Ten external ECGs for workflow calibration only; not a powered efficacy sample.'
SOURCE_ACCESS_POLICY = 'Exact archived generator ECG image only; global amplitude normalization and calibration limitations recorded.'
ALLOW_PARTIAL = True  # Sample jointly successful cases; native workflow records coverage/exclusions.
print('Review directory:', REVIEW_DIR)
print('Planned reports:', N_QUERIES * len(SELECTED_MODELS) * len(ARMS))


In [ ]:
if not REVIEWER_EXPERTISE.strip() or '[' in REVIEWER_EXPERTISE:
    raise ValueError('Enter actual reviewer expertise and study involvement before creating the package.')
archive = archive_folder(CFG)
if not read(archive / 'status.json')['complete']:
    raise ValueError('Finish BR04C before clinical preparation')
# This validates extraction provenance, original waveforms and exact archived PNGs.
provider = verified_provider(CFG)
identity = dict(config_sha256=sha(CONFIG_PATH), archive_protocol_sha256=sha(archive/'protocol.json'),
    archive_manifest_sha256=sha(archive/'manifest.json'), models=list(SELECTED_MODELS), arms=list(ARMS),
    n_queries=N_QUERIES, seed=CFG['seed'], stage=STAGE, reviewer_id=REVIEWER_ID,
    reviewer_expertise=REVIEWER_EXPERTISE, sample_size_justification=SAMPLE_SIZE_JUSTIFICATION,
    source_access_policy=SOURCE_ACCESS_POLICY, allow_partial=ALLOW_PARTIAL,
    sources={m: {'protocol': sha(f/'protocol.json'), 'reports': sha(f/'reports.jsonl')} for m,f in FOLDERS.items()})
REVIEW_DIR.parent.mkdir(parents=True, exist_ok=True)
with lock(REVIEW_DIR.parent / (REVIEW_VERSION + '_prepare.lock')):
    if REVIEW_DIR.exists():
        marker = REVIEW_DIR / 'PRIVATE/shaoxing_archive_preparation.json'
        if not marker.is_file() or read(marker) != identity:
            raise ValueError('Existing package differs or preparation was interrupted. Preserve it; inspect before choosing a fresh version.')
        print('Existing package preserved:', progress(REVIEW_DIR))
    else:
        def archive_provider(config):
            if config != CFG: raise ValueError('Different config requested')
            return provider
        # Native prepare checks each selected report against exact image/tensor hashes.
        # Only image sourcing changes; no checks or clinical judgments are bypassed.
        with patch.object(queries, 'image_provider', side_effect=archive_provider):
            result = prepare(CFG, FOLDERS, REVIEW_DIR, n_queries=N_QUERIES, arms=ARMS,
                seed=CFG['seed'], reviewer_id=REVIEWER_ID, reviewer_expertise=REVIEWER_EXPERTISE,
                stage=STAGE, sample_size_justification=SAMPLE_SIZE_JUSTIFICATION,
                allow_partial=ALLOW_PARTIAL, source_access_policy=SOURCE_ACCESS_POLICY)
        immutable(REVIEW_DIR / 'PRIVATE/shaoxing_archive_preparation.json', identity)
        print(result)
print(progress(REVIEW_DIR))


In [ ]:
# Create an editable copy; never overwrite human entries or the protected native template.
public = REVIEW_DIR / 'PUBLIC'
completed = public / 'reference_cases_completed.json'
if not completed.exists():
    cases = read(public / 'reference_cases.json')
    form = [dict(case_id=r['case_id'], image=r['image'], source_image_sha256=sha(public/r['image']),
                 reviewer_id=REVIEWER_ID, interpretable=None,
                 independent_ecg_interpretation='', limitations='') for r in cases]
    with completed.open('x', encoding='utf-8') as stream:
        json.dump(form, stream, indent=2)
    print('Created blank human-entry form:', completed)
else:
    print('Existing human entries preserved:', completed)
instructions = public / 'SHAOXING_HUMAN_REVIEW_INSTRUCTIONS.txt'
if instructions.exists() and instructions.read_bytes() != GUIDE.read_bytes():
    raise ValueError('Existing reviewer instructions differ; preserve and inspect version history.')
if not instructions.exists(): instructions.write_bytes(GUIDE.read_bytes())
print('DOWNLOAD:', public)
print('First round: images + reference_cases_completed.json + instructions only.')
print('UPLOAD completed form to:', completed)
print('KEEP PRIVATE HIDDEN:', REVIEW_DIR / 'PRIVATE')


## Continue using the existing import and review notebooks

1. **BR07A**: set `REVIEW_DIR` to the directory printed above and import `PUBLIC/reference_cases_completed.json` after all independent ECG readings are complete.
2. **BR08**: set `CONFIG_PATH` to the Shaoxing child config and `REVIEW_VERSION` to the same version. Release reports only after references are frozen. Rate reports against the ECG and your first-round interpretation. Use BR08 directly or the existing offline forms and **BR08A** import, setting `REVIEW_DIR` to this package.
3. In a later session, perform same-reviewer adjudication of flagged discrepancies. Use **BR08B** with this `REVIEW_DIR` to import completed decisions. Retain the original ratings; unresolved cases stay unresolved.
4. **BR09**: same Shaoxing `CONFIG_PATH` and `REVIEW_VERSION`; analyze both initial and adjudicated outcomes. Ten ECGs remain calibration only.

Default Biowulf package:
`/path/to/repository/outputs/real_run_001/experiment02_v1/context_fit/v1/shaoxing/clinical_review/shaoxing/calibration_v1`

Label-mapping review is still pending unless separately completed and documented. Human report review does not automatically unlock BR05A automated label scoring.
